In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv('customer_shopping_behavior.csv')

In [ ]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [ ]:
df.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [ ]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.mean()))

In [ ]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)' : 'purchase_amount'})

In [ ]:
labels = ['Young Adult','Adult','Middle-aged','Senior']
df['age_group'] = pd.qcut (df['age'],q = 4,labels=labels)

In [ ]:
df[['age','age_group']].head()

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged


In [ ]:
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [ ]:
df = df.drop('promo_code_used',axis = 1)

In [ ]:
import sqlite3
conn = sqlite3.connect('customer_data.db')


In [ ]:
df.to_sql(
    name='sales_data',
    con=conn,
    if_exists='replace',
    index=False
)

3900

In [ ]:
pd.read_sql_query("SELECT * FROM sales_data", conn)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,32,Venmo,Weekly,Adult,7
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,41,Bank Transfer,Bi-Weekly,Middle-aged,14
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,24,Venmo,Quarterly,Middle-aged,90
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,24,Venmo,Weekly,Adult,7


Q1. Total revenue generated by customers based on gender

In [ ]:
pd.read_sql_query("SELECT gender,SUM(purchase_amount) as revenue FROM sales_data GROUP BY gender", conn)

,gender,revenue
0,Female,75191
1,Male,157890


Q2. Which customers used a discount but shopped more than average purchase amount

In [ ]:
spd.read_sql_query("SELECT customer_id, purchase_amount FROM sales_data WHERE discount_applied = 'Yes' AND purchase_amount >= (SELECT AVG(purchase_amount) FROM sales_data)", conn)

,customer_id,purchase_amount
0,2,64
1,3,73
2,4,90
3,7,85
4,9,97
...,...,...
834,1667,64
835,1671,73
836,1673,73
837,1674,62


Q3. Find top 5 products with highest average review rating

In [ ]:
pd.read_sql_query("SELECT item_purchased, AVG(review_rating) as avg_rating FROM sales_data GROUP BY item_purchased ORDER BY avg_rating DESC LIMIT 5",conn)

,item_purchased,avg_rating
0,Gloves,3.860785
1,Sandals,3.844336
2,Boots,3.818707
3,Hat,3.801104
4,Skirt,3.784946


Q4.Compare avg purchase amount between standard and express shipping

In [ ]:
pd.read_sql_query("Select shipping_type,AVG(purchase_amount) From sales_data WHERE shipping_type in ('Standard', 'Express') Group by shipping_type",conn)

,shipping_type,AVG(purchase_amount)
0,Express,60.475232
1,Standard,58.460245


Q5. Do subscribed customers spend more? Compare average spend and total revenue
between subscribers and non-subscribers.

In [ ]:
pd.read_sql_query("SELECT subscription_status,Count(customer_id) as total_customers , AVG(purchase_amount) as avg_spend, SUM(purchase_amount) as total_revenue FROM sales_data GROUP BY subscription_status Order by total_revenue,avg_spend", conn)

,subscription_status,total_customers,avg_spend,total_revenue
0,Yes,1053,59.491928,62645
1,No,2847,59.865121,170436


Q6. Which 5 products have the highest percentage of purchases with discounts applied?

In [ ]:
pd.read_sql_query("SELECT item_purchased, (100*SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) / COUNT(*)) AS discount_percentage FROM sales_data GROUP BY item_purchased ORDER BY discount_percentage DESC LIMIT 5",conn)

,item_purchased,discount_percentage
0,Hat,50
1,Sneakers,49
2,Coat,49
3,Sweater,48
4,Pants,47


Q7. Segment customers into New, Returning, and Loyal based on their total
 number of previous purchases, and show the count of each segment.

In [ ]:
pd.read_sql_query("SELECT customer_id, previous_purchases, CASE WHEN previous_purchases = 1 THEN 'New' WHEN previous_purchases between 2 and 10 THEN 'Returning' else 'LOYAL' END AS segment FROM sales_data",conn)

,customer_id,previous_purchases,segment
0,1,14,LOYAL
1,2,2,Returning
2,3,23,LOYAL
3,4,49,LOYAL
4,5,31,LOYAL
...,...,...,...
3895,3896,32,LOYAL
3896,3897,41,LOYAL
3897,3898,24,LOYAL
3898,3899,24,LOYAL


Q8. What are the top 3 most purchased products within each category?

In [ ]:
pd.read_sql_query("""WITH item_counts AS (
    SELECT category,
           item_purchased,
           COUNT(customer_id) AS total_orders,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY COUNT(customer_id) DESC) AS item_rank
    FROM sales_data
    GROUP BY category, item_purchased
)
SELECT item_rank,category, item_purchased, total_orders
FROM item_counts
WHERE item_rank <=3;""",conn)

,item_rank,category,item_purchased,total_orders
0,1,Accessories,Jewelry,171
1,2,Accessories,Sunglasses,161
2,3,Accessories,Belt,161
3,1,Clothing,Pants,171
4,2,Clothing,Blouse,171
5,3,Clothing,Shirt,169
6,1,Footwear,Sandals,160
7,2,Footwear,Shoes,150
8,3,Footwear,Sneakers,145
9,1,Outerwear,Jacket,163


Q9. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subscribe?

In [ ]:
pd.read_sql_query("""SELECT subscription_status,
       COUNT(customer_id) AS repeat_buyers
FROM sales_data
WHERE previous_purchases > 5
GROUP BY subscription_status;""",conn)

,subscription_status,repeat_buyers
0,No,2518
1,Yes,958


Q10. What is the revenue contribution of each age group?

In [ ]:
pd.read_sql_query("""SELECT
    age_group,
    SUM(purchase_amount) AS total_revenue
FROM sales_data
GROUP BY age_group
ORDER BY total_revenue desc;""",conn)

,age_group,total_revenue
0,Young Adult,62143
1,Middle-aged,59197
2,Adult,55978
3,Senior,55763


In [ ]:
final_df = pd.read_sql_query(
    "SELECT * FROM sales_data",
    conn
)

In [ ]:
final_df.to_csv("cleaned_sales_data.csv", index=False)

In [ ]:
from google.colab import files
files.download("cleaned_sales_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>